In [12]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Optimizing Shuffles")
    .master("spark://016e64429d30:7077")
    # CPU cores Spark can use across the entire application. This is a cluster-wide limit for Spark app
    .config("spark.cores.max", 16)
    # Number of cores per executor
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark

In [13]:
# Check Spark defaultParallelism

spark.sparkContext.defaultParallelism

16

In [14]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)  # Disable runtime optimization (static execution plan)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)  # Prevent dynamic reduction of shuffle partitions
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)  # Disable broadcast joins (force shuffle-based joins)

In [13]:
# Read EMP CSV file with 10M records

_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load("/data/input/employee_records.csv")

In [5]:
# Find out avg salary as per dept
from pyspark.sql.functions import avg

emp_avg = emp.groupBy("department_id").agg(avg("salary").alias("avg_sal"))
#emp_avg.show()

In [6]:
# Write data for performance Benchmarking

emp_avg.write.format("noop").mode("overwrite").save()

# .format("noop") – "noop" is a special format in Spark.
# It essentially does nothing; it’s mostly used for testing or debugging write operations without actually writing to disk or any external storage.

In [7]:
# Check Spark Shuffle Partition setting

spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [17]:
# 1st stage will be 16 tasks to read the data
# 2nd stage will be 200 tasks for groupBy aggregation; By default spark uses 200 shuffle partitions

In [21]:
from pyspark.sql.functions import spark_partition_id

emp.withColumn("partition_id", spark_partition_id()).where("partition_id=0").show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|partition_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|           0|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|           0|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|           0|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|           0|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffany

In [8]:
# Executors: 4 executors with IDs 0, 1, 2, 3.
# Total Tasks per Executor: Each executor ran 4 tasks.
# Shuffle Write Size / Records: Each executor wrote 2.8 KiB / 40 records.
# Input Records per Executor: Between ~217,000 to 260,000 records read per executor.

# In Spark, shuffle write records correspond to shuffle partitions. Each task writes shuffle data for all output partitions assigned to it.
# Here, each executor ran 4 tasks, and overall, 4 executors × 4 tasks = 16 tasks total.
# Each executor writes 40 shuffle records in total across those 4 tasks, so:
# Shuffle records per task = 40 records / 4 tasks = 10 records per task
# So each task writes shuffle data for about 10 partitions.

# Each Spark task processes one input partition containing multiple departments (e.g., 10 departments per task).
# Before writing shuffle data, Spark performs partial aggregation (map-side combine) within each task, summarizing data (like averages) per department.
# This aggregation reduces the number of records each task writes to shuffle.
# You have 4 executors × 4 tasks each = 16 tasks total, and each executor writes shuffle files for about 40 output partitions.
# So, the 40 shuffle write records per executor represent the aggregated data partitions written after combining.
# These shuffle files are then sent over the network and read by the next stage for further processing.

In [20]:
# By default, Spark sets shuffle partitions to 200 (spark.sql.shuffle.partitions=200).
# If your job only needs fewer partitions (less parallelism), many tasks may be small or empty, so they do very little or skip processing.
# These extra tasks still consume CPU and scheduling overhead without doing meaningful work.
# This overhead leads to longer execution times and wasted resources.

# Reduce the shuffle partitions to better match your data size and cluster capacity
spark.conf.set("spark.sql.shuffle.partitions", 20)

In [21]:
spark.conf.get("spark.sql.shuffle.partitions")

'20'

In [14]:
emp_avg.write.format("noop").mode("overwrite").save()

In [22]:
# Creating partitioned data

emp_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"
df = spark.read.format("csv").schema(emp_schema).option("header", True).load("/data/input/employee_records.csv")
df.write \
  .format("csv") \
  .mode("overwrite") \
  .option("header", True) \
  .partitionBy("department_id") \
  .save("/home/jupyter/pyspark_notes_and_codes/data/output/emp_partitioned")

In [23]:
# Read the partitioned data

_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp_part = spark.read.format("csv").schema(_schema).option("header", True).load("/data/output/emp_partitioned/")

In [24]:
from pyspark.sql.functions import avg

emp_part_avg = emp_part.groupBy("department_id").agg(avg("salary").alias("avg_salary"))
#emp_part_avg.show(10)

emp_part_avg.write.format("noop").mode("overwrite").save()

In [ ]:
# Reading partitioned data is faster because Spark can use partition pruning:
#   it reads only the folders relevant to the filter instead of the entire dataset.
# Less shuffle write happens because data is already organized by the partition column,
#   so operations like groupBy or join on that column require less data movement across nodes.

In [25]:
spark.stop()